In [ ]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("..") / "data" / "processed" / "bike_share_daily.parquet"
df = pd.read_parquet(DATA_PATH)
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df[df["city"] == "San Francisco"].head()

In [ ]:
df[df["city"] == "New York City"].head()

## San Francisco Growth Analysis — MTC February 2023 Investment

Looking at whether Bay Wheels ridership and station-network size grew after the Metropolitan Transportation Commission's February 2023 investment.

**Data caveat:** `bike_share_daily.parquet` has a gap for San Francisco — no rows for **January–May 2024** (the file was built from two separate download windows: 2023 full-year, then July 2024 onward). `2024-06` is also a partial/boundary month (19 rows, ~1 trip each) and should be excluded from any trend read. This doesn't affect the before/after comparison below since neither comparison window touches the gap, but it does mean the full monthly trend chart has a real hole in it, not a data quality artifact to ignore.

In [ ]:
import matplotlib.pyplot as plt

sf = df[df["city"] == "San Francisco"].copy()
sf["month"] = sf["date"].dt.to_period("M")

# 2024-06 is a one-day boundary artifact (19 rows, ~1 trip each) from the
# two-part download window — exclude it from the trend so it doesn't read
# as a ridership collapse.
monthly_trips = (
    sf[sf["month"] != pd.Period("2024-06")]
    .groupby("month")["trips"]
    .sum()
)
monthly_trips

In [ ]:
months_str = monthly_trips.index.astype(str)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(months_str, monthly_trips.to_numpy(), marker="o")
ax.axvline(x="2023-02", color="red", linestyle="--", label="Feb 2023 (MTC investment)")
ax.axvspan("2024-01", "2024-05", color="grey", alpha=0.15, label="Data gap (Jan–May 2024)")
ax.set_title("San Francisco (Bay Wheels) — Total Trips per Month")
ax.set_xlabel("Month")
ax.set_ylabel("Total trips")
ax.tick_params(axis="x", rotation=90)
ax.legend()
plt.tight_layout()
plt.show()

### Average daily trips per station: right after Feb 2023 vs most recent

Comparing two equal-length (6-month) windows — the six months immediately after the Feb 2023 investment (Mar–Aug 2023) vs the six most recent available months (Jan–Jun 2026) — so the comparison isn't skewed by window size. Per-station, not just total trips, so network expansion (more stations) doesn't automatically inflate the "growth" story.

In [ ]:
post_window = pd.period_range("2023-03", "2023-08", freq="M")
recent_window = pd.period_range("2026-01", "2026-06", freq="M")

comparison_rows = []
for label, window in [
    ("Post-Feb-2023 (Mar-Aug 2023)", post_window),
    ("Most recent (Jan-Jun 2026)", recent_window),
]:
    sub = sf[sf["month"].isin(window)]
    daily = sub.groupby("date").agg(trips=("trips", "sum"), stations=("station_name", "nunique"))
    daily["trips_per_station"] = daily["trips"] / daily["stations"]
    comparison_rows.append(
        {
            "window": label,
            "avg_daily_total_trips": daily["trips"].mean(),
            "avg_active_stations_per_day": daily["stations"].mean(),
            "avg_daily_trips_per_station": daily["trips_per_station"].mean(),
        }
    )

comparison = pd.DataFrame(comparison_rows).set_index("window").round(2)
comparison

### Station network size: Feb 2023 vs now

In [ ]:
feb2023_stations = set(sf.loc[sf["month"] == pd.Period("2023-02"), "station_name"].unique())
latest_month = sf["month"].max()
latest_stations = set(sf.loc[sf["month"] == latest_month, "station_name"].unique())

net_new = latest_stations - feb2023_stations
lost = feb2023_stations - latest_stations
pct_growth = 100 * (len(latest_stations) - len(feb2023_stations)) / len(feb2023_stations)

station_summary = pd.DataFrame(
    {
        "metric": [
            "Stations active in Feb 2023",
            f"Stations active in {latest_month}",
            "Net new stations",
            "Stations no longer active",
            "Net network growth (%)",
        ],
        "value": [
            len(feb2023_stations),
            len(latest_stations),
            len(net_new),
            len(lost),
            round(pct_growth, 1),
        ],
    }
).set_index("metric")
station_summary

## Findings

**Yes — both ridership and network size grew after the February 2023 investment, and the growth compounded over time rather than being a one-time bump.**

- **Ridership per station roughly grew:** average daily trips per station went from **~14.2** (Mar–Aug 2023, right after the investment) to **~24.3** (Jan–Jun 2026, most recent) — a **+71% increase**. This is per-station, so it isn't just "more stations means more trips."
- **Total daily ridership more than doubled:** average daily total trips rose from **~6,150** to **~13,740** — a **+124% increase**, reflecting both more trips per station and a larger network.
- **The network expanded:** San Francisco had **509 active stations** in Feb 2023 vs **638** in the most recent month (Jun 2026) — **214 net-new stations, 85 stations no longer active**, for **+25% net growth** in station count.
- **The monthly trend chart shows sustained growth, not a spike:** trips climb steadily from early 2023 through mid-2023, and the post-gap period (Jul 2024 onward) continues that same upward trajectory into 2026, rather than flattening or reverting.

**Caveat:** there's a real gap in this dataset for Jan–May 2024 (see the note above), so we can't see exactly how the trend evolved through that stretch. The comparison above avoids the gap entirely (Mar–Aug 2023 vs Jan–Jun 2026), so the headline growth numbers aren't affected — but a reviewer will reasonably ask about that missing stretch, so it's worth closing before this goes in a report.